El Hareket Siniflandirma

In [1]:
!pip install pillow

In [2]:

import random
import shutil
from pathlib import Path

# -------------------------------
# 1. AYARLAR
# -------------------------------

SOURCE_DIR = Path("datasets/hands")
OUTPUT_DIR = Path("dataset_split")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_STATE = 42

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

# -------------------------------
# 2. KONTROLLER
# -------------------------------

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Veri seti bulunamadi: {SOURCE_DIR}"
    )

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Cikti klasoru zaten mevcut: {OUTPUT_DIR}. "
        "Veri kaybini onlemek icin once kontrol et."
    )

if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-9:
    raise ValueError("Oranlarin toplami 1 olmali.")

random.seed(RANDOM_STATE)

# -------------------------------
# 3. SINIFLARI BUL
# -------------------------------

class_dirs = sorted(
    [p for p in SOURCE_DIR.iterdir() if p.is_dir()]
)

if not class_dirs:
    raise ValueError("Sinif klasoru bulunamadi.")

total_images = 0

# -------------------------------
# 4. HER SINIFI AYRI AYRI BOL
# -------------------------------

for class_dir in class_dirs:

    class_name = class_dir.name

    images = [
        p for p in class_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    random.shuffle(images)

    n = len(images)

    if n < 3:
        raise ValueError(
            f"{class_name} sinifinda yeterli goruntu yok: {n}"
        )

    # Veri sayilarini hesapla
    train_count = int(n * TRAIN_RATIO)
    val_count = int(n * VAL_RATIO)

    test_count = n - train_count - val_count

    # Verileri ayir
    train_images = images[:train_count]

    val_images = images[
        train_count:train_count + val_count
    ]

    test_images = images[
        train_count + val_count:
    ]

    splits = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    # -------------------------------
    # 5. KLASORLERI OLUSTUR VE KOPYALA
    # -------------------------------

    print(f"\nSinif: {class_name}")

    for split_name, split_images in splits.items():

        target_dir = OUTPUT_DIR / split_name / class_name

        target_dir.mkdir(parents=True, exist_ok=True)

        for image_path in split_images:
            shutil.copy2(
                image_path,
                target_dir / image_path.name
            )

        print(
            f"{split_name}: {len(split_images)} goruntu"
        )

    total_images += n

# -------------------------------
# 6. SONUC RAPORU
# -------------------------------

print("\n" + "=" * 40)
print("VERI SETI BOLME TAMAMLANDI")
print("=" * 40)

for split_name in ["train", "val", "test"]:

    split_dir = OUTPUT_DIR / split_name

    count = sum(
        1 for p in split_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    percentage = count / total_images * 100

    print(
        f"{split_name.upper():5} : "
        f"{count:6} goruntu (%{percentage:.2f})"
    )

print(f"\nToplam goruntu: {total_images}")
print(f"Cikti klasoru: {OUTPUT_DIR.resolve()}")



Sinif: closedFist
train: 148 goruntu
val: 31 goruntu
test: 33 goruntu

Sinif: fingerCircle
train: 163 goruntu
val: 34 goruntu
test: 36 goruntu

Sinif: fingerSymbols
train: 370 goruntu
val: 79 goruntu
test: 80 goruntu

Sinif: multiFingerBend
train: 48 goruntu
val: 10 goruntu
test: 11 goruntu

Sinif: openPalm
train: 128 goruntu
val: 27 goruntu
test: 29 goruntu

Sinif: semiOpenFist
train: 252 goruntu
val: 54 goruntu
test: 55 goruntu

Sinif: semiOpenPalm
train: 237 goruntu
val: 51 goruntu
test: 52 goruntu

Sinif: singleFingerBend
train: 167 goruntu
val: 35 goruntu
test: 37 goruntu

VERI SETI BOLME TAMAMLANDI
TRAIN :   1513 goruntu (%69.82)
VAL   :    321 goruntu (%14.81)
TEST  :    333 goruntu (%15.37)

Toplam goruntu: 2167
Cikti klasoru: C:\Users\Beyza Nur\Desktop\Projelerim\python\dataset_split


In [3]:
import ultralytics

In [9]:
ultralytics.checks()

Ultralytics 8.4.198  Python-3.14.10 torch-2.15.0+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
Setup complete  (12 CPUs, 7.3 GB RAM, 85.7/476.0 GB disk)


In [10]:

from ultralytics import YOLO

# YOLO11 Classification modelini indir
model = YOLO("yolo11n-cls.pt")

print("Model başarıyla yüklendi!")

# Model bilgilerini kontrol et
print(model.info())

Model başarıyla yüklendi!
YOLO11n-cls summary: 86 layers, 2,812,104 parameters, 0 gradients, 3.3 GFLOPs
(86, 2812104, 0, 3.31530496)


In [32]:
results = model.train(
    data="hands_datasets/train",
    epochs=50,
    imgsz=224,
    batch=8,
    patience=3,
    project="hands_project",
    name="hands_training"
)

metrics = model.val(
    data="hands_datasets/val",
    split="val"
)
print("Validation metrics:", metrics)

New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.198  Python-3.14.10 torch-2.15.0+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=hands_datasets/train, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Beyza Nu

In [1]:
import numpy as np
import cv2
import ultralytics
from ultralytics import YOLO
#from google.colab.patches import cv2_imshow 

img_path="C:\\Users\\Beyza Nur\\Desktop\\Projelerim\\python\\hands_datasets\\train\\closedFist\\IMG_20220501_155119.jpg"
model_path="C:\\Users\\Beyza Nur\\Desktop\\Projelerim\\python\\runs\\classify\\hands_project\\hands_training-2\\weights\\best.pt"
model=YOLO(model_path)
results=model(img_path)

class_dict= results[0].names
class_prob= results[0].probs.data.tolist()

print("Class probabilities:", class_prob[np.argmax(class_prob)])   
print("Predicted class:", class_dict[np.argmax(class_prob)])


image 1/1 C:\Users\Beyza Nur\Desktop\Projelerim\python\hands_datasets\train\closedFist\IMG_20220501_155119.jpg: 224x224 closedFist 0.99, semiOpenFist 0.00, semiOpenPalm 0.00, fingerSymbols 0.00, fingerCircle 0.00, 25.1ms
Speed: 4.0ms preprocess, 25.1ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Class probabilities: 0.9937248826026917
Predicted class: closedFist


In [8]:
img=cv2.imread(img_path)
text=f"{class_dict[np.argmax(class_prob)]} : {class_prob[np.argmax(class_prob)]:.2f}"
cv2.putText(img, text, (10, 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 25, 120), 1)
cv2.imshow("Prediction", img)
cv2.waitKey(0)
cv2.destroyAllWindows()
